# BWPS-clean: clean-up of Bertolo et al. (2026) data

A minority of the predicted syllable types in BWPS have a markedly longer duration and higher durational variability (standard deviation) than other syllable types. This is inconsistent with the other datasets, and visual inspection of the timelines suggests these may be motif interruptions rather than genuine syllable types.

We therefore create a second version of the dataset, `bwps-clean`, in which we remove such syllables and replace them with chunk boundaries. A label is removed if, in that bird,
- it occurs only once or twice,
- it has a long average duration (at least 250 ms) and is rare (fewer than 20 occurrences), or
- it has a long average duration and an unusually large standard deviation (modified z-score > 7, twice the conventional outlier threshold of 3.5).

The remaining labels are then relabeled alphabetically, without refitting the HMMs. These criteria were chosen to approximate the results of a manual inspection of the timelines of all birds.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

import sys
sys.path.append('../')

from src.data import save_units
from src.metrics.permutation import shuffle_labels_within_chunk

ROOT_DIR = Path('.').resolve().parent
DATASET_ID = 'bwps' # Bertolo et al. 2026 dataset
OUTPUT_DIR = ROOT_DIR / 'data'

In [2]:
from scipy.stats import median_abs_deviation

def modified_zscore(values):
    values = np.asarray(values)
    return (values - np.median(values)) / median_abs_deviation(values, scale='normal')

def relabel_alphabetically(labels):
    alphabet = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    unique_labels = np.sort(labels.unique())
    mapping = { lab: alphabet[i] for i, lab in enumerate(unique_labels)}
    return [mapping[lab] for lab in labels]

def segment_chunks_at_match(units, match_fn, drop=True, relabel=True, renumber=True, label_col='label_pred'):
    # Add a column indicating when a chunk starts
    units = units.copy()
    chunk_idxs = units['chunk_idx'].values
    chunk_start = np.concat([np.array([True]), chunk_idxs[1:] != chunk_idxs[:-1]])
    units['is_chunk_start'] = chunk_start

    new_units = []
    chunk_idx = 0
    for _, unit in units.iterrows():
        unit = unit.to_dict()
        is_match = match_fn(unit)
        if is_match or unit['is_chunk_start']:
            # New chunk
            start = 0
            chunk_idx += 1

        unit['start'] = start
        unit['chunk_idx'] = chunk_idx

        if not (is_match and drop):
            new_units.append(unit)
            start += unit['duration']
    new_units = pd.DataFrame(new_units).drop(columns=['is_chunk_start'])

    # Relabel syllables so that labels form a contiguous sequence A, B, C, ...
    if relabel:
        new_labels = relabel_alphabetically(new_units[label_col])
        new_units[label_col] = new_labels

    # Renumber chunks 1, 2, 3, ...
    if renumber:
        unique_chunks = new_units['chunk_idx'].unique()
        chunks_map = dict(zip(unique_chunks, range(1, len(unique_chunks)+1)))
        new_units['chunk_idx'] = [chunks_map[i] for i in new_units['chunk_idx']]

    return new_units

def segment_chunks_at_labels(units, labels, label_col='label_pred', **kwargs):
    match_fn = lambda unit: unit[label_col] in labels
    return segment_chunks_at_match(units, match_fn, **kwargs)

In [3]:
def get_exceptional_labels(
    units, 
    std_threshold=7, 
    dur_threshold=0.25,
    rare_freq=20,
    label_col='label_pred'
):
    """
    Identify exceptional labels: 
    - labels that occur 1-2 times
    - labels that are long (>250ms) on average and rare (freq < 20) 
    - labels that are long (>250ms) on average and have a very big standard deviation 
      (modified z-score > 7)
    """
    dur_by_label = units.groupby(label_col)['duration']
    
    # Mean duration
    stats = pd.DataFrame()
    stats['dur_mean'] = dur_by_label.mean()
    stats['dur_mean_z'] = modified_zscore(stats['dur_mean'])
    
    # Std of durations
    stats['dur_std'] = dur_by_label.std()
    std_zscores = modified_zscore(stats['dur_std'].dropna())
    stats.loc[stats['dur_std'].dropna().index, 'dur_std_z'] = std_zscores

    # Frequency
    stats['freq'] = units[label_col].value_counts()
    stats['freq_z'] = modified_zscore(stats['freq'])
    
    # Criteria
    big_var = stats['dur_std_z'] > std_threshold
    is_long = stats['dur_mean'] >= dur_threshold
    is_unicum = stats['freq'] <= 2
    is_rare = stats['freq'] < rare_freq
    stats['exceptional'] = (big_var & is_long) | is_unicum | (is_rare & is_long)

    return stats.index[stats['exceptional']].values, stats

In [4]:
# Load the original bwps dataset
from src.data import load_units

units = load_units('bwps', 'syll_ioi', use_milliseconds=False)

In [5]:
# Segment at exceptional units
all_new_units = []
for bird_id in units['bird_id'].unique():
    orig_units = units[units['bird_id'] == bird_id]
    labels, _ = get_exceptional_labels(orig_units)
    new_units = segment_chunks_at_labels(orig_units, labels)
    all_new_units.append(new_units)

all_new_units = pd.concat(all_new_units, axis=0).reset_index().drop(columns=['index'])
all_new_units['bird_id'] = [id.replace('bwps-', 'bwps-clean-') for id in all_new_units['bird_id']]

from src.rhythm_types import drop_short_chunks

# A chunk needs at least 2 intervals to derive any dyad from it later.
n_before = all_new_units.groupby(['bird_id', 'chunk_idx']).ngroups
all_new_units = drop_short_chunks(all_new_units)
n_after = all_new_units.groupby(['bird_id', 'chunk_idx']).ngroups
print(f"dropped {n_before - n_after} of {n_before} chunks with fewer than 2 intervals")

dropped 59 of 2021 chunks with fewer than 2 intervals


In [6]:
# Shuffled baseline for plots (see preprocess-ns), shuffled from 'label_pred'.
# Recomputed rather than taken from bwps, since resegmentation changed the chunks.
rng = np.random.default_rng(0)
all_new_units['label_shuffle'] = shuffle_labels_within_chunk(
    all_new_units, rng, by=['bird_id', 'chunk_idx'], label_col='label_pred',
)['label_pred']

In [7]:
# Write units to disk
save_units(all_new_units, 'syll_ioi', f"{DATASET_ID}-clean", OUTPUT_DIR)